Level 2: Data Cleaning & Feature Engineering

Import Libraries

In [2]:
import pandas as pd
import numpy as np

Load Dataset

In [3]:
df = pd.read_csv("Dataset1.csv")

First 5 rows

In [4]:
df.head()

,SN,Train_No,Station_Code,1A,2A,3A,SL,Station_Name,Route_Number,Arrival_time,Departure_Time,Distance
0,1,107,SWV,100,100,100,100,SAWANTWADI R,1,00:00:00,10:25:00,0
1,2,107,THVM,260,228,196,164,THIVIM,1,11:06:00,11:08:00,32
2,3,107,KRMI,345,296,247,198,KARMALI,1,11:28:00,11:30:00,49
3,4,107,MAO,490,412,334,256,MADGOAN JN.,1,12:10:00,00:00:00,78
4,1,108,MAO,100,100,100,100,MADGOAN JN.,1,00:00:00,20:30:00,0


Task 2.1: Handle Missing Values and Remove Duplicate Records
Objective

In [5]:
# Check missing values
print(df.isnull().sum())

# Check total missing values
print("Total Missing Values:", df.isnull().sum().sum())

# Check duplicate rows
print("Duplicate Rows:", df.duplicated().sum())

# Remove duplicates if any
df = df.drop_duplicates()

SN                0
Train_No          0
Station_Code      0
1A                0
2A                0
3A                0
SL                0
Station_Name      0
Route_Number      0
Arrival_time      0
Departure_Time    0
Distance          0
dtype: int64
Total Missing Values: 0
Duplicate Rows: 0


Task 2.2: Standardize Arrival and Departure Time Formats

In [6]:
df['Arrival_time'] = pd.to_datetime(
    df['Arrival_time'],
    format='%H:%M:%S',
    errors='coerce'
)

df['Departure_Time'] = pd.to_datetime(
    df['Departure_Time'],
    format='%H:%M:%S',
    errors='coerce'
)

print(df[['Arrival_time','Departure_Time']].dtypes)

Arrival_time      datetime64[us]
Departure_Time    datetime64[us]
dtype: object


Task 2.3: Calculate Total Journey Duration for Each Train

In [7]:
# Sort data
df = df.sort_values(['Train_No', 'SN'])

duration_data = []

for train, group in df.groupby('Train_No'):

    start_time = group.iloc[0]['Departure_Time']
    end_time = group.iloc[-1]['Arrival_time']

    duration = (end_time - start_time).total_seconds() / 3600

    if duration < 0:
        duration += 24

    duration_data.append([train, duration])

journey_duration = pd.DataFrame(
    duration_data,
    columns=['Train_No', 'Journey_Duration_Hours']
)

journey_duration.head()

,Train_No,Journey_Duration_Hours
0,107,1.750000
1,108,1.916667
2,128,22.083333
3,290,8.000000
4,401,12.500000


Task 2.4: Create Features Such as Total Distance and Number of Stops

Feature 1: Total Distance

In [9]:
distance_feature = (
    df.groupby('Train_No')['Distance']
      .max()
      .reset_index(name='Total_Distance')
)

distance_feature.head()

,Train_No,Total_Distance
0,107,78
1,108,83
2,128,978
3,290,2694
4,401,1618


Feature 2: Number of Stops

In [10]:
stops_feature = (
    df.groupby('Train_No')
      .size()
      .reset_index(name='Number_of_Stops')
)

stops_feature.head()

,Train_No,Number_of_Stops
0,107,4
1,108,4
2,128,22
3,290,14
4,401,12


Merge Features

In [11]:
final_features = (
    journey_duration
    .merge(distance_feature, on='Train_No')
    .merge(stops_feature, on='Train_No')
)

final_features.head()

,Train_No,Journey_Duration_Hours,Total_Distance,Number_of_Stops
0,107,1.750000,78,4
1,108,1.916667,83,4
2,128,22.083333,978,22
3,290,8.000000,2694,14
4,401,12.500000,1618,12


In [12]:
final_features.to_csv("final_features.csv", index=False)

Conclusion

In this level, the dataset was successfully cleaned and transformed. Missing values and duplicate records were checked, time columns were standardized, journey duration was calculated, and new features such as Total Distance and Number of Stops were created. These processed features will be used in the next phase of Exploratory Data Analysis (EDA) and predictive modeling.